# Experiment log (FNAL-AI4QC-GRU)

Created: 2026-10-03. Running record of experiments in this repo; results are added as they land.

## 2026-10-03: d=9 MWPM-fix queue (pre-registered)

Pre-registration, written before any run: [`prereg/mwpm_fixes_d9_2026-10.md`](../prereg/mwpm_fixes_d9_2026-10.md).
Driver and job table: `experiments/mwpm_fixes_d9_2026-10/` (`driver.py`, `jobs.tsv`); results land in `results.tsv` there.


## 2026-10-21: GRU depth scaling at d=7 (pre-registered)

L x GRU u70, L = 1-10, plus single layers matched in parameters to L = 4 (u172) and L = 8 (u255), after an LR stability scan at L = 1 and 10.
Pre-registration: [`prereg/depth_scaling_d7_2026-10.md`](../prereg/depth_scaling_d7_2026-10.md).
Runs: `experiments/depth_scaling_d7_2026-10/` (`status.log`, `lr_scan/`, `depth_sweep/`).
Results: `GRU_depth_scaling_runs_<date>.csv` at the repo root; figures `figures/depth_scaling.py`.

### Result, 2026-10-23

Sweep LR 1e-3 (L=10 stable at 1e-3 and 2e-3, unstable at 3e-3, diverged at 5e-3).
Eval block [15.2M, 17M), 1.8M shots, MWPM 0.004291 on the same shots.

| L | parameters | selected epoch | p_L | p_L / MWPM |
|---|---|---|---|---|
| 1 | 25,061 | 78 | 0.014812 | 3.452 |
| 2 (kd pilot hl7_2x70) | 54,671 | 80 | 0.005432 | 1.266 |
| 3 | 84,281 | 78 | 0.005264 | 1.227 |
| 4 | 113,891 | 78 | 0.003636 | 0.847 |
| 5 | 143,501 | 80 | 0.003462 | 0.807 |
| 6 | 173,111 | 77 | 0.003431 | 0.800 |
| 7 | 202,721 | 77 | 0.003336 | 0.777 |
| 8 | 232,331 | 78 | 0.003332 | 0.777 |
| 9 | 261,941 | 79 | 0.003137 | 0.731 |
| 10 | 291,551 | 79 | 0.003055 | 0.712 |
| 1 x u172 (matches L=4) | 114,209 | 78 | 0.005544 | 1.292 |
| 1 x u255 (matches L=8) | 232,816 | 78 | 0.003649 | 0.850 |

- L* = 5 by the pre-registered rule: L=5 -> 6 gains 0.9% with McNemar p 0.4. The stack first
  beats MWPM at L=4 (0.847). Past L=5 the gains are small and uneven (L=7 -> 8 gains 0.1%,
  p 0.92; L=8 -> 9 gains 5.9%, p 2.4e-08), and L=10 is 0.712.
- Depth, not parameter count: each stack beats the single layer of the same size, L=4 (0.847)
  against u172 (1.292) with p 9e-321, L=8 (0.777) against u255 (0.850) with p 5e-15.
- No degradation at large L: training loss falls monotonically with depth, 0.0401 at L=1 to
  0.0087 at L=10.
- No run is converged at 80 epochs: every last-20-epoch slope is negative with t from -6.6 to
  -16.0, so these ratios are upper bounds.
- The 2026-10-22 21:19 UTC reboot killed L=5 to L=10 mid-run; they were resumed from the last
  complete epoch with the Adam state and the same shot stream (prereg amendment).


## 2026-10-28: kd5_u140 quantization study toward hls4ml (pre-registered rule; results added as they land)

Folder: `runs/qat_kd5_u140/` (plan and quantizer mapping in `NOTES.md`, summary in `SUMMARY.md`). Branch `qat-kd5-u140`.

### Selection rule, written before any QAT result (2026-10-28 ~13:00Z)

Eval block only ([15.2M, 17M), 1.8M shots, MWPM decoded on the same shots). A configuration is acceptable if its LER <= 1.05 x the FP32 parent's LER and its LER < MWPM's LER. The candidate is the acceptable configuration with the fewest total bits (weight bits + state bits), ties broken by fewer state bits. If nothing below 16 + 16 bits is acceptable, that is reported as the result. The sealed block and the held-out seeds are not read.

### Result, 2026-10-28 19:05Z (Phase 3 hls4ml C-sim and EAF synthesis still running)

FP32 parent reproduced exactly on the eval block: 9,267 errors (0.005148), MWPM 13,627 (0.007571), ratio 0.680.

Profiled integer bits (max over 1.8M shots, + 1 headroom): GRU kernel 7, recurrent kernel 8, bias 5, Dense kernel 5, Dense bias 2, GRU accumulator 9, h 3, tables 2, Dense accumulator 7.

Knees (PTQ, selection rule): weights 12 bits (1.018x FP32; 10 bits 1.66x), state 8 bits (1.027x; 6 bits 2.52x).

Candidate under the rule: QAT, 12-bit weights, 6-bit state (h and tables), 24-bit accumulators: 0.005353 = 1.040x FP32, 0.707x MWPM (seed 0); 3-seed mean 1.035x, every seed acceptable. Its neighbor one state bit lower (w12_s5) fails on every seed (mean 1.078x). w10_s8 also reaches 18 bits (mean 1.032x) and loses on the state-bit tie-break. No QAT point at 8 weight bits or fewer is acceptable (w8 1.115x).

Open questions: weight outliers limit low-bit weights (recurrent max 52, 99th percentile 3.8); early stopping ended runs before the LR decayed; accumulator width not swept; synthesis decides w12_s6 vs w10_s8 on cost. Full table, decisions made unattended and open questions: `runs/qat_kd5_u140/SUMMARY.md`.
